# AutoBio-Inspired Computer Vision Self-Learning Notebook

This notebook is a self-learning module for computer vision (CV) students. It is inspired by the AutoBio paper, which introduces a simulation and benchmark for robotic automation in biology laboratories.

The notebook follows one rule throughout:

**Concept first, code second.**

The learning goal is not to reproduce the full AutoBio simulator. Instead, the goal is to understand CV-relevant ideas behind the benchmark:

- visual reasoning from multiple camera views
- object localization in lab scenes
- slot and rack reasoning
- transparent-liquid level estimation
- UI reading and instruction grounding
- task scoring and benchmark-style evaluation
- why precision manipulation is hard for vision-language-action systems

All code uses synthetic toy data so that the notebook can run without downloading external datasets.

## 0. Learning Tasks

By the end of this notebook, students should be able to complete the following tasks:

1. Explain how a robotic lab task can be decomposed into biological primitives, motion primitives, and CV/VLA capabilities.
2. Generate simple synthetic lab images for CV experiments.
3. Detect rack slots and tube positions using image-processing features.
4. Estimate liquid level from a transparent tube-like object.
5. Read simplified instrument-panel values and compare them with a language instruction.
6. Implement benchmark-style scoring for binary success and relative progress.
7. Analyze why easy, medium, and hard robotic lab tasks stress different visual reasoning skills.

In [ ]:
# If this cell fails, install the missing packages in your environment.
# The notebook is designed to use standard scientific Python packages.

import math
import random
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

try:
    import cv2
    HAS_CV2 = True
except Exception:
    HAS_CV2 = False

from PIL import Image, ImageDraw

np.random.seed(7)
random.seed(7)

print("OpenCV available:", HAS_CV2)

## 1. Concept: From Biological Protocols to CV Tasks

AutoBio frames biology-lab automation as a hierarchy:

- **Biological primitives** describe what the experiment needs, such as transfer, combination, measurement, conditioning, preservation, and separation.
- **Motion primitives** describe how the robot acts, such as reach-and-align, relocate, actuate, and transfer.
- **VLA capabilities** describe what the model must infer, such as precision control, instruction following, and visual reasoning.

For a CV-focused notebook, we can reinterpret these ideas as visual tasks:

| Lab action | CV question |
|---|---|
| Pick up a centrifuge tube | Where is the tube? |
| Transfer a tube to row/column target | Which slot is the target? |
| Aspirate liquid | Where is the liquid surface? |
| Load centrifuge rotor symmetrically | Which slot is opposite the occupied slot? |
| Operate a thermal mixer panel | What values are shown on the display? |

The key point is that a robotic benchmark is also a visual benchmark. The robot cannot act correctly unless the visual state is understood correctly.

In [ ]:
# A small utility for consistent image display.

def show_image(img, title=None, figsize=(5, 5)):
    """Display a grayscale or RGB image."""
    plt.figure(figsize=figsize)
    if isinstance(img, Image.Image):
        img = np.array(img)
    if img.ndim == 2:
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(img)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()

## 2. Concept: Synthetic Lab Scene Generation

Professional lab datasets are expensive. Simulation is useful because it can generate many labeled scenes with controlled randomization.

In AutoBio, simulation randomization is used to test whether a policy generalizes beyond memorized trajectories. For CV learning, we can use randomization to create toy images where the ground-truth labels are known.

In the next code cell, we generate a simplified rack scene:

- a rectangular rack
- multiple circular slots
- one tube placed in a random slot
- a target row and column
- ground-truth coordinates for evaluation

This toy scene supports tasks similar to "pick up the tube" and "transfer the tube to row $r$, column $c$."

In [ ]:
@dataclass
class RackScene:
    image: np.ndarray
    slot_centers: list
    tube_slot: tuple
    target_slot: tuple
    rows: int
    cols: int

def generate_rack_scene(rows=4, cols=6, image_size=(420, 560), noise_std=4):
    """Generate a synthetic rack scene with one tube and one target slot."""
    h, w = image_size
    img = np.ones((h, w, 3), dtype=np.uint8) * 245

    rack_x0, rack_y0 = 80, 80
    rack_w, rack_h = w - 160, h - 160

    img[rack_y0:rack_y0+rack_h, rack_x0:rack_x0+rack_w] = (210, 210, 210)

    slot_centers = []
    margin_x = rack_w / (cols + 1)
    margin_y = rack_h / (rows + 1)

    tube_slot = (random.randrange(rows), random.randrange(cols))
    target_slot = (random.randrange(rows), random.randrange(cols))
    while target_slot == tube_slot:
        target_slot = (random.randrange(rows), random.randrange(cols))

    if HAS_CV2:
        for r in range(rows):
            for c in range(cols):
                cx = int(rack_x0 + (c + 1) * margin_x)
                cy = int(rack_y0 + (r + 1) * margin_y)
                slot_centers.append(((r, c), (cx, cy)))
                cv2.circle(img, (cx, cy), 18, (90, 90, 90), 2)
                cv2.circle(img, (cx, cy), 13, (235, 235, 235), -1)

        tube_center = dict(slot_centers)[tube_slot]
        cv2.circle(img, tube_center, 17, (40, 95, 210), -1)
        cv2.circle(img, tube_center, 20, (20, 50, 120), 2)

        target_center = dict(slot_centers)[target_slot]
        cv2.rectangle(img, (target_center[0]-25, target_center[1]-25),
                      (target_center[0]+25, target_center[1]+25), (230, 80, 40), 2)
    else:
        pil = Image.fromarray(img)
        draw = ImageDraw.Draw(pil)
        for r in range(rows):
            for c in range(cols):
                cx = int(rack_x0 + (c + 1) * margin_x)
                cy = int(rack_y0 + (r + 1) * margin_y)
                slot_centers.append(((r, c), (cx, cy)))
                draw.ellipse((cx-18, cy-18, cx+18, cy+18), outline=(90, 90, 90), width=2)
                draw.ellipse((cx-13, cy-13, cx+13, cy+13), fill=(235, 235, 235))
        tube_center = dict(slot_centers)[tube_slot]
        draw.ellipse((tube_center[0]-17, tube_center[1]-17,
                      tube_center[0]+17, tube_center[1]+17), fill=(40, 95, 210))
        target_center = dict(slot_centers)[target_slot]
        draw.rectangle((target_center[0]-25, target_center[1]-25,
                        target_center[0]+25, target_center[1]+25), outline=(230, 80, 40), width=2)
        img = np.array(pil)

    noise = np.random.normal(0, noise_std, img.shape)
    img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    return RackScene(img, slot_centers, tube_slot, target_slot, rows, cols)

scene = generate_rack_scene()
show_image(scene.image, title=f"Tube slot={scene.tube_slot}, target slot={scene.target_slot}", figsize=(7, 5))

## 3. Concept: Object Localization by Color and Geometry

A VLA system may use deep neural networks, but classical CV can still teach the core idea: perception turns pixels into structured state.

For the rack scene, the structured state is:

- tube center: $(x, y)$
- nearest rack slot: $(r, c)$
- target slot: $(r, c)$

The simplified detector below uses color thresholding. This is not meant to be a robust real-world method. It is a learning tool for understanding how perception errors affect downstream robotic decisions.

In [ ]:
def detect_blue_tube(img):
    """Detect the synthetic blue tube and return its center."""
    arr = np.asarray(img)

    r = arr[:, :, 0].astype(np.int16)
    g = arr[:, :, 1].astype(np.int16)
    b = arr[:, :, 2].astype(np.int16)
    mask = (b > 140) & (r < 100) & (g < 140)

    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None, mask.astype(np.uint8) * 255

    center = (float(xs.mean()), float(ys.mean()))
    return center, mask.astype(np.uint8) * 255

def nearest_slot(center, slot_centers):
    """Map a detected center to the nearest known rack slot."""
    if center is None:
        return None
    x, y = center
    best_slot = None
    best_dist = float("inf")
    for slot, (cx, cy) in slot_centers:
        dist = math.sqrt((x - cx) ** 2 + (y - cy) ** 2)
        if dist < best_dist:
            best_dist = dist
            best_slot = slot
    return best_slot

center, mask = detect_blue_tube(scene.image)
pred_slot = nearest_slot(center, scene.slot_centers)

show_image(mask, title="Detected tube mask")
print("Detected center:", center)
print("Predicted tube slot:", pred_slot)
print("Ground-truth tube slot:", scene.tube_slot)

### Student Task 1: Stress-Test Tube Localization

Change the parameters of `generate_rack_scene()`:

- increase `noise_std`
- increase `rows` and `cols`
- modify tube color inside the generator
- generate 100 scenes and compute accuracy

Question: When does the simple detector fail, and why would a learned model be useful?

In [ ]:
def evaluate_tube_detector(n=100, noise_std=4):
    """Evaluate the tube detector over many randomized scenes."""
    correct = 0
    failures = 0

    for _ in range(n):
        s = generate_rack_scene(noise_std=noise_std)
        center, _ = detect_blue_tube(s.image)
        pred = nearest_slot(center, s.slot_centers)
        if pred is None:
            failures += 1
        elif pred == s.tube_slot:
            correct += 1

    return {
        "n": n,
        "noise_std": noise_std,
        "accuracy": correct / n,
        "detection_failures": failures
    }

for noise in [0, 4, 10, 20, 35]:
    print(evaluate_tube_detector(n=100, noise_std=noise))

## 4. Concept: Slot Reasoning and Language Grounding

The AutoBio transfer task uses an instruction such as:

"Move the centrifuge tube to row `{target_row}`, column `{target_col}`."

This requires language grounding: a symbolic instruction must be mapped to a visual location.

In this toy version, the instruction is already structured as a row-column pair. The visual reasoning step is to find the corresponding slot center.

The important idea is that perception and language are not separate. A wrong row or column interpretation can cause a physically precise robot to complete the wrong task.

In [ ]:
def target_center_from_instruction(scene, target_row, target_col):
    """Convert a row-column instruction into a pixel coordinate."""
    lookup = dict(scene.slot_centers)
    return lookup[(target_row, target_col)]

target_row, target_col = scene.target_slot
target_center = target_center_from_instruction(scene, target_row, target_col)

print("Instruction:")
print(f"Move the centrifuge tube to row {target_row}, column {target_col}.")
print("Target center:", target_center)

vis = scene.image.copy()
if HAS_CV2:
    cv2.drawMarker(vis, target_center, (255, 0, 0), markerType=cv2.MARKER_CROSS, markerSize=35, thickness=3)
else:
    pil = Image.fromarray(vis)
    draw = ImageDraw.Draw(pil)
    x, y = target_center
    draw.line((x-20, y, x+20, y), fill=(255, 0, 0), width=3)
    draw.line((x, y-20, x, y+20), fill=(255, 0, 0), width=3)
    vis = np.array(pil)

show_image(vis, title="Grounded target slot")

### Student Task 2: Create Instruction Errors

Modify `target_row` or `target_col` by $+1$ or $-1$. Then visualize the new target.

Question: How large is the visual difference between a correct slot and a neighboring wrong slot? Why is this dangerous in precision manipulation?

In [ ]:
row_offset = 0
col_offset = 0

wrong_row = min(max(target_row + row_offset, 0), scene.rows - 1)
wrong_col = min(max(target_col + col_offset, 0), scene.cols - 1)
wrong_center = target_center_from_instruction(scene, wrong_row, wrong_col)

print("Original target:", scene.target_slot, target_center)
print("Modified target:", (wrong_row, wrong_col), wrong_center)

## 5. Concept: Symmetry Reasoning for Rotor Loading

The AutoBio hard task "Load centrifuge rotor" asks the robot to insert a second tube into the slot that is symmetrically opposite the currently placed tube.

This is a visual reasoning task because the system must:

1. detect the occupied slot,
2. infer the rotor center,
3. reason over circular symmetry,
4. select the opposite slot.

For a rotor with $N$ evenly spaced slots, the opposite slot index is:

$$
j = (i + N/2) \bmod N
$$

This formula only works when $N$ is even.

In [ ]:
@dataclass
class RotorScene:
    image: np.ndarray
    centers: list
    occupied_index: int
    opposite_index: int
    rotor_center: tuple

def generate_rotor_scene(n_slots=12, image_size=(480, 480), radius=150):
    """Generate a synthetic centrifuge rotor with one occupied slot."""
    h, w = image_size
    cx, cy = w // 2, h // 2
    img = np.ones((h, w, 3), dtype=np.uint8) * 245

    occupied = random.randrange(n_slots)
    opposite = (occupied + n_slots // 2) % n_slots

    centers = []
    if HAS_CV2:
        cv2.circle(img, (cx, cy), radius + 35, (190, 190, 190), -1)
        cv2.circle(img, (cx, cy), radius + 35, (100, 100, 100), 3)
        for i in range(n_slots):
            angle = 2 * math.pi * i / n_slots
            sx = int(cx + radius * math.cos(angle))
            sy = int(cy + radius * math.sin(angle))
            centers.append((sx, sy))
            cv2.circle(img, (sx, sy), 18, (250, 250, 250), -1)
            cv2.circle(img, (sx, sy), 18, (80, 80, 80), 2)
        cv2.circle(img, centers[occupied], 16, (45, 105, 210), -1)
        cv2.circle(img, centers[opposite], 24, (230, 80, 40), 2)
    else:
        pil = Image.fromarray(img)
        draw = ImageDraw.Draw(pil)
        draw.ellipse((cx-radius-35, cy-radius-35, cx+radius+35, cy+radius+35), fill=(190, 190, 190), outline=(100, 100, 100), width=3)
        for i in range(n_slots):
            angle = 2 * math.pi * i / n_slots
            sx = int(cx + radius * math.cos(angle))
            sy = int(cy + radius * math.sin(angle))
            centers.append((sx, sy))
            draw.ellipse((sx-18, sy-18, sx+18, sy+18), fill=(250, 250, 250), outline=(80, 80, 80), width=2)
        ox, oy = centers[occupied]
        draw.ellipse((ox-16, oy-16, ox+16, oy+16), fill=(45, 105, 210))
        tx, ty = centers[opposite]
        draw.ellipse((tx-24, ty-24, tx+24, ty+24), outline=(230, 80, 40), width=2)
        img = np.array(pil)

    return RotorScene(img, centers, occupied, opposite, (cx, cy))

rotor = generate_rotor_scene()
show_image(rotor.image, title=f"Occupied={rotor.occupied_index}, opposite={rotor.opposite_index}")

In [ ]:
def infer_opposite_slot(occupied_index, n_slots):
    """Infer the opposite rotor slot for an even number of slots."""
    if n_slots % 2 != 0:
        raise ValueError("The number of slots must be even for exact opposite-slot reasoning.")
    return (occupied_index + n_slots // 2) % n_slots

pred_opposite = infer_opposite_slot(rotor.occupied_index, len(rotor.centers))

print("Predicted opposite slot:", pred_opposite)
print("Ground-truth opposite slot:", rotor.opposite_index)
print("Correct:", pred_opposite == rotor.opposite_index)

### Student Task 3: Rotate the Rotor

Generate a rotor scene with a random global rotation angle. Then update the code so that the reasoning still works.

Question: Which parts of the reasoning should be index-based, and which parts should be geometry-based?

## 6. Concept: Transparent Liquid and Liquid-Level Sensing

The AutoBio paper highlights transparent materials and liquid reasoning as important challenges. Transparent tubes and liquids are hard for simple renderers and hard for perception systems.

In a simplified CV task, liquid level can be estimated from a tube image by finding a horizontal boundary.

For a tube of height $H$, if the detected liquid surface is at vertical pixel coordinate $y_s$ and the tube bottom is $y_b$, the approximate fill ratio is:

$$
\text{fill ratio} = \frac{y_b - y_s}{H}
$$

This toy formula assumes the tube is vertical and the camera is front-facing.

In [ ]:
@dataclass
class TubeLiquidScene:
    image: np.ndarray
    true_fill_ratio: float
    tube_box: tuple

def generate_tube_liquid_scene(image_size=(420, 260), fill_ratio=None, noise_std=3):
    """Generate a synthetic transparent tube with a colored liquid region."""
    if fill_ratio is None:
        fill_ratio = random.uniform(0.15, 0.85)

    h, w = image_size
    img = np.ones((h, w, 3), dtype=np.uint8) * 250
    x0, y0, x1, y1 = 90, 40, 170, 360
    tube_h = y1 - y0
    liquid_top = int(y1 - fill_ratio * tube_h)

    if HAS_CV2:
        cv2.rectangle(img, (x0, y0), (x1, y1), (170, 170, 170), 3)
        cv2.rectangle(img, (x0+3, liquid_top), (x1-3, y1-3), (105, 180, 230), -1)
        cv2.line(img, (x0+3, liquid_top), (x1-3, liquid_top), (30, 110, 170), 2)
        cv2.rectangle(img, (x0+12, y0+8), (x0+25, y1-8), (255, 255, 255), -1)
    else:
        pil = Image.fromarray(img)
        draw = ImageDraw.Draw(pil)
        draw.rectangle((x0, y0, x1, y1), outline=(170, 170, 170), width=3)
        draw.rectangle((x0+3, liquid_top, x1-3, y1-3), fill=(105, 180, 230))
        draw.line((x0+3, liquid_top, x1-3, liquid_top), fill=(30, 110, 170), width=2)
        draw.rectangle((x0+12, y0+8, x0+25, y1-8), fill=(255, 255, 255))
        img = np.array(pil)

    noise = np.random.normal(0, noise_std, img.shape)
    img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    return TubeLiquidScene(img, fill_ratio, (x0, y0, x1, y1))

tube_scene = generate_tube_liquid_scene()
show_image(tube_scene.image, title=f"True fill ratio={tube_scene.true_fill_ratio:.2f}", figsize=(4, 6))

In [ ]:
def estimate_fill_ratio(img, tube_box):
    """Estimate liquid fill ratio by detecting blue-ish pixels inside the tube."""
    x0, y0, x1, y1 = tube_box
    crop = img[y0:y1, x0:x1]

    r = crop[:, :, 0].astype(np.int16)
    g = crop[:, :, 1].astype(np.int16)
    b = crop[:, :, 2].astype(np.int16)

    mask = (b > 150) & (g > 120) & (r < 170)
    ys, xs = np.where(mask)

    if len(ys) == 0:
        return 0.0, None

    liquid_top_local = int(np.min(ys))
    fill_ratio = ((y1 - y0) - liquid_top_local) / (y1 - y0)
    return float(fill_ratio), y0 + liquid_top_local

pred_fill, pred_surface_y = estimate_fill_ratio(tube_scene.image, tube_scene.tube_box)

print("True fill ratio:", round(tube_scene.true_fill_ratio, 3))
print("Estimated fill ratio:", round(pred_fill, 3))
print("Absolute error:", round(abs(tube_scene.true_fill_ratio - pred_fill), 3))

vis = tube_scene.image.copy()
if pred_surface_y is not None:
    x0, y0, x1, y1 = tube_scene.tube_box
    if HAS_CV2:
        cv2.line(vis, (x0, pred_surface_y), (x1, pred_surface_y), (255, 0, 0), 3)
    else:
        pil = Image.fromarray(vis)
        ImageDraw.Draw(pil).line((x0, pred_surface_y, x1, pred_surface_y), fill=(255, 0, 0), width=3)
        vis = np.array(pil)

show_image(vis, title="Estimated liquid surface", figsize=(4, 6))

### Student Task 4: Add Hard Cases

Make the liquid-level task harder by adding:

- a slanted liquid surface
- transparent highlights
- darker shadows
- partial occlusion by a gripper
- a tilted tube

Question: Which hard cases break this simple algorithm? Which cases might require a neural segmentation model?

## 7. Concept: Instrument UI Reading and Instruction Following

The AutoBio hard task "Operate thermal mixer panel" requires a robot to set speed, temperature, and time using visual feedback from an instrument display.

This combines:

- reading the current visual state
- parsing the target instruction
- comparing current and target values
- deciding which buttons to press

A simplified target instruction can be represented as:

$$
\mathbf{u}_{target} = [\text{rpm}, \text{temperature}, \text{time}]
$$

The current display can be represented as:

$$
\mathbf{u}_{current} = [\text{rpm}, \text{temperature}, \text{time}]
$$

The control error is:

$$
\mathbf{e} = \mathbf{u}_{target} - \mathbf{u}_{current}
$$

In real systems, this error must be reduced through physical button presses and visual feedback.

In [ ]:
@dataclass
class PanelScene:
    image: np.ndarray
    current_values: dict
    target_values: dict

def generate_panel_scene(current=None, target=None, image_size=(260, 620)):
    """Generate a simplified instrument panel with text values."""
    if current is None:
        current = {
            "rpm": random.randrange(200, 1601, 100),
            "temp": random.randrange(20, 81, 5),
            "time": random.randrange(10, 301, 10),
        }
    if target is None:
        target = {
            "rpm": random.randrange(200, 1601, 100),
            "temp": random.randrange(20, 81, 5),
            "time": random.randrange(10, 301, 10),
        }

    img = Image.new("RGB", image_size[::-1], (235, 235, 235))
    draw = ImageDraw.Draw(img)

    draw.rounded_rectangle((40, 40, 580, 220), radius=18, fill=(60, 65, 70), outline=(20, 20, 20), width=3)
    draw.rectangle((70, 70, 550, 150), fill=(15, 40, 35), outline=(150, 190, 170), width=2)

    text = f"RPM {current['rpm']:04d}   TEMP {current['temp']:02d}C   TIME {current['time']:03d}s"
    draw.text((85, 98), text, fill=(180, 255, 210))

    labels = ["RPM+", "RPM-", "TEMP+", "TEMP-", "TIME+", "TIME-"]
    x = 75
    for label in labels:
        draw.rounded_rectangle((x, 170, x+70, 205), radius=8, fill=(200, 200, 205), outline=(80, 80, 80))
        draw.text((x+10, 180), label, fill=(20, 20, 20))
        x += 80

    return PanelScene(np.array(img), current, target)

panel = generate_panel_scene()
show_image(panel.image, title="Synthetic thermal mixer panel", figsize=(10, 4))

print("Current values:", panel.current_values)
print("Target instruction:", panel.target_values)

In [ ]:
def compute_panel_error(current, target):
    """Compute target-current errors for panel parameters."""
    return {key: target[key] - current[key] for key in target}

def propose_button_presses(current, target, step_sizes=None):
    """Create a simple button-press plan to move current values toward target values."""
    if step_sizes is None:
        step_sizes = {"rpm": 100, "temp": 5, "time": 10}

    plan = []
    for key in ["rpm", "temp", "time"]:
        diff = target[key] - current[key]
        step = step_sizes[key]
        if diff == 0:
            continue
        button = f"{key.upper()}+" if diff > 0 else f"{key.upper()}-"
        n_presses = abs(diff) // step
        plan.extend([button] * int(n_presses))
    return plan

error = compute_panel_error(panel.current_values, panel.target_values)
plan = propose_button_presses(panel.current_values, panel.target_values)

print("Error:", error)
print("Proposed button presses:", plan)
print("Number of presses:", len(plan))

### Student Task 5: Close the Loop

Write a function that applies one button press, updates the current values, redraws the panel, and repeats until the target is reached.

Question: Why does closed-loop visual feedback matter more than open-loop control for this task?

In [ ]:
def apply_button(current, button, step_sizes=None):
    """Apply one symbolic button press to the current panel state."""
    if step_sizes is None:
        step_sizes = {"RPM": 100, "TEMP": 5, "TIME": 10}

    new_current = dict(current)
    if button.endswith("+"):
        key = button[:-1]
        sign = 1
    else:
        key = button[:-1]
        sign = -1

    map_key = {"RPM": "rpm", "TEMP": "temp", "TIME": "time"}[key]
    new_current[map_key] += sign * step_sizes[key]
    return new_current

current = dict(panel.current_values)
for press in plan[:5]:
    current = apply_button(current, press)

print("State after first five planned presses:", current)
print("Remaining error:", compute_panel_error(current, panel.target_values))

## 8. Concept: Benchmark Scoring

AutoBio uses binary success for most tasks and a relative progress score for the thermal mixer panel task.

A binary score is simple:

$$
S =
\begin{cases}
1, & \text{task succeeds} \\
0, & \text{task fails}
\end{cases}
$$

For panel operation, a relative progress score can capture partial success:

$$
S = \sum_{i=1}^{3} w_i \max \left(1 - \frac{|final_i - target_i|}{|initial_i - target_i|}, 0 \right)
$$

The score is usually normalized to the interval $[0, 1]$ by choosing weights such that:

$$
\sum_{i=1}^{3} w_i = 1
$$

This is useful when a task is too hard for most policies to fully complete.

In [ ]:
def binary_success(pred_slot, target_slot):
    """Return a binary success score."""
    return int(pred_slot == target_slot)

def relative_progress_score(initial, final, target, weights=None):
    """Compute a weighted relative progress score for panel values."""
    keys = ["rpm", "temp", "time"]
    if weights is None:
        weights = {"rpm": 1/3, "temp": 1/3, "time": 1/3}

    score = 0.0
    for key in keys:
        denom = abs(initial[key] - target[key])
        if denom == 0:
            term = 1.0 if final[key] == target[key] else 0.0
        else:
            term = max(1.0 - abs(final[key] - target[key]) / denom, 0.0)
        score += weights[key] * term
    return float(score)

initial = panel.current_values
target = panel.target_values
partial_final = current

print("Initial:", initial)
print("Partial final:", partial_final)
print("Target:", target)
print("Relative progress score:", round(relative_progress_score(initial, partial_final, target), 3))

### Student Task 6: Compare Scoring Rules

Create three final panel states:

1. perfect final state
2. halfway final state
3. worse-than-initial final state

Compute the relative progress score for each.

Question: What does the progress score reward that binary success does not?

In [ ]:
perfect = dict(target)
halfway = {key: int(round((initial[key] + target[key]) / 2)) for key in target}
worse = {key: initial[key] - (target[key] - initial[key]) for key in target}

for name, final_state in [("perfect", perfect), ("halfway", halfway), ("worse", worse)]:
    print(name, final_state, "score =", round(relative_progress_score(initial, final_state, target), 3))

## 9. Concept: Difficulty Levels and Failure Modes

AutoBio organizes tasks into easy, medium, and hard levels.

A CV-centered interpretation is:

| Difficulty | Example task | Main CV challenge |
|---|---|---|
| Easy | close/open thermal cycler lid | object state and trajectory following |
| Easy | pick up centrifuge tube | basic object localization |
| Medium | unscrew centrifuge tube cap | precise pose and dual-arm coordination |
| Medium | aspirate with pipette | liquid-level perception and tool alignment |
| Medium | transfer centrifuge tube | language-grounded slot localization |
| Hard | operate thermal mixer panel | UI reading, feedback, instruction following |
| Hard | load centrifuge rotor | symmetry reasoning and precise placement |
| Hard | screw on centrifuge tube cap | alignment-sensitive manipulation |

The important lesson is that task difficulty is not only about action length. A short task can be hard if it requires precise visual reasoning.

In [ ]:
tasks = [
    {"task": "close lid", "precision": 1, "language": 1, "visual_reasoning": 1},
    {"task": "pick tube", "precision": 2, "language": 1, "visual_reasoning": 2},
    {"task": "transfer tube", "precision": 3, "language": 3, "visual_reasoning": 2},
    {"task": "aspirate liquid", "precision": 4, "language": 2, "visual_reasoning": 4},
    {"task": "operate panel", "precision": 4, "language": 5, "visual_reasoning": 5},
    {"task": "load rotor", "precision": 4, "language": 3, "visual_reasoning": 5},
    {"task": "screw cap", "precision": 5, "language": 2, "visual_reasoning": 4},
]

for item in tasks:
    item["total"] = item["precision"] + item["language"] + item["visual_reasoning"]

for item in sorted(tasks, key=lambda x: x["total"]):
    print(item)

## 10. Concept: Geometry Behind Simulated Lab Mechanisms

The AutoBio paper includes several physics and geometry ideas. For CV students, these formulas are useful because they connect rendered visual states to underlying 3D structure.

### 10.1 Helix model for threaded caps

A circular helix can model a screw thread:

$$
H(t; r_1, p) =
\begin{bmatrix}
r_1 \cos t \\
r_1 \sin t \\
pt
\end{bmatrix}
$$

Here, $r_1$ is the helix radius and $p$ controls vertical pitch.

For a point $P \in \mathbb{R}^3$, an approximate distance to an unbounded helix can be written as:

$$
t_0 = \operatorname{atan2}(P_y, P_x)
$$

$$
k = \left\lfloor \frac{P_z - t_0p}{2\pi p} \right\rceil
$$

$$
\operatorname{SDF}(P) =
\left\| H(2k\pi + t_0; r_1, p) - P \right\|_2
$$

This kind of representation helps simulate cap-thread contact without a dense mesh.

In [ ]:
def helix_points(r1=1.0, p=0.08, turns=4, n=400):
    """Generate 3D helix points."""
    t = np.linspace(0, 2 * np.pi * turns, n)
    x = r1 * np.cos(t)
    y = r1 * np.sin(t)
    z = p * t
    return x, y, z

x, y, z = helix_points()

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
ax.plot(x, y, z)
ax.set_title("Circular helix for a simplified thread")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
plt.show()

### 10.2 Detent mechanism for click positions

A detent mechanism models discrete click positions, such as a stepped knob.

A simplified passive force model is:

$$
f(q, \dot{q}) = -k(q - q_j) - \lambda \dot{q}
$$

The nearest click index is:

$$
j = \arg\min_i |q - q_i|,\quad i = 0,\ldots,n-1
$$

For CV, the visual state of the knob may be continuous, but the task state is often discrete.

In [ ]:
def nearest_click(q, n=8):
    """Return the nearest discrete click position on a circular knob."""
    positions = np.linspace(0, 2 * np.pi, n, endpoint=False)
    diffs = np.abs(np.angle(np.exp(1j * (q - positions))))
    j = int(np.argmin(diffs))
    return j, positions[j]

q = 1.7
j, qj = nearest_click(q, n=8)
print("Current angle:", round(q, 3))
print("Nearest click index:", j)
print("Nearest click angle:", round(qj, 3))

### 10.3 Relative progress for UI manipulation

The panel score used earlier is also a perception metric. If the display is read incorrectly, the controller may believe progress has been made when it has not.

This is a common benchmark issue: the measured state must correspond to the real task state.

In [ ]:
true_final = dict(partial_final)
misread_final = dict(partial_final)
misread_final["rpm"] = target["rpm"]

print("True score:", round(relative_progress_score(initial, true_final, target), 3))
print("Score with misread RPM:", round(relative_progress_score(initial, misread_final, target), 3))

## 11. Mini-Project: Build a CV Benchmark Card

Choose one of the following toy tasks:

1. Tube slot localization
2. Rack transfer target grounding
3. Rotor opposite-slot reasoning
4. Liquid-level estimation
5. Panel parameter matching

Create a benchmark card with:

- task name
- input image type
- ground-truth label
- success metric
- likely failure modes
- one easy version
- one hard version

Then implement at least one baseline algorithm and evaluate it on 100 randomized scenes.

In [ ]:
benchmark_card = {
    "task_name": "Tube slot localization",
    "input": "RGB image of a rack with one tube",
    "ground_truth": "row-column index of the occupied slot",
    "metric": "slot accuracy",
    "easy_version": "low noise, distinct tube color, fixed camera",
    "hard_version": "high noise, occlusion, distractor objects, changed lighting",
    "failure_modes": [
        "tube color threshold fails",
        "nearest-slot assignment is wrong",
        "occlusion hides the tube",
        "lighting changes make the tube look different"
    ]
}

benchmark_card

## 12. Summary

This notebook converted an AutoBio-inspired robotics benchmark into CV learning tasks.

Main takeaways:

- Simulation can produce labeled visual data for self-learning.
- Robotic manipulation tasks often hide CV subproblems.
- Easy-looking tasks may become hard because of precision, transparency, symmetry, or UI feedback.
- Benchmark scoring should match the task goal and should reward meaningful partial progress when full success is rare.
- Classical CV baselines are useful for understanding the structure of the problem, even when deep learning is needed for real-world robustness.

## References

- AutoBio: A Simulation and Benchmark for Robotic Automation in Digital Biology Laboratory.
- MIDeL-style learning structure: concept explanations paired with executable notebook examples.